# Lesson 3: Memory
So far, every `invoke` started from nothing. The agent forgot everything between calls. A real user says *"I have $5,000"* once and expects the agent to remember. This lesson fixes that.

Run the cells top to bottom with **Shift+Enter**.

## Step 0: Setup
The tools now live in `stock_advisor/tools.py`, so we import them instead of copying them.

In [1]:
import sys
sys.path.insert(0, '..')   # lets the notebook find the stock_advisor folder

from dotenv import load_dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from stock_advisor.tools import ALL_TOOLS

load_dotenv()
MODEL = 'groq:openai/gpt-oss-120b'

## Step 1: Prove the problem
This agent has no memory. We tell it a fact, then ask for it back in a new call.

In [2]:
forgetful = create_agent(model=MODEL, tools=[], system_prompt='You are a helpful assistant. Be brief.')

forgetful.invoke({'messages': [{'role': 'user', 'content': 'My budget is $5,000.'}]})
reply = forgetful.invoke({'messages': [{'role': 'user', 'content': 'What is my budget?'}]})
print(reply['messages'][-1].content)

I don’t have any information about your budget. Could you provide more details or let me know what you’re planning?


It cannot know. The second call never saw the first. The model has no memory of its own. **We** have to send it the earlier messages every time.

## Step 2: Add memory with a checkpointer
A *checkpointer* saves the conversation after every step. A *thread id* is the name of one conversation. Same thread id means same conversation.

In [3]:
agent = create_agent(
    model=MODEL,
    tools=[],
    system_prompt='You are a helpful assistant. Be brief.',
    checkpointer=InMemorySaver(),   # this one line adds memory
)

alice = {'configurable': {'thread_id': 'alice'}}

agent.invoke({'messages': [{'role': 'user', 'content': 'My budget is $5,000.'}]}, config=alice)
reply = agent.invoke({'messages': [{'role': 'user', 'content': 'What is my budget?'}]}, config=alice)
print(reply['messages'][-1].content)

Your budget is $5,000.


## Step 3: Different thread, different conversation
Each user gets their own thread id. Their conversations never mix.

In [4]:
bob = {'configurable': {'thread_id': 'bob'}}

reply = agent.invoke({'messages': [{'role': 'user', 'content': 'What is my budget?'}]}, config=bob)
print(reply['messages'][-1].content)

I’m not sure—could you share the details of your budget or what you’re planning?


## Step 4: Look at what was saved
You can read a thread's saved messages at any time.

In [5]:
state = agent.get_state(alice)
for m in state.values['messages']:
    print(m.type.upper(), ':', m.content)

HUMAN : My budget is $5,000.
AI : Got it! What are you looking to spend the $5,000 on—travel, a home project, a special event, or something else?
HUMAN : What is my budget?
AI : Your budget is $5,000.


## Step 5: The stock advisor with memory
Now the real tools plus memory. We state a budget and a style once, then ask follow-up questions without repeating ourselves.

In [9]:
from IPython.display import display, Markdown

advisor = create_agent(
    model=MODEL,
    tools=ALL_TOOLS,
    system_prompt=(
        "You are a careful stock research assistant. The user gives company names, not tickers. "
        "Find the ticker first, then use the tools for evidence.\n"
        "Rules: only state numbers that came from a tool; say 'I don't have data' instead of guessing; "
        "remember what the user told you earlier (budget, risk level); "
        "keep answers under 250 words and do not add extra sections the user did not ask for; "
        "always say this is research, not financial advice."
    ),
    checkpointer=InMemorySaver(),
)

me = {'configurable': {'thread_id': 'me'}}

def ask(text):
    result = advisor.invoke({'messages': [{'role': 'user', 'content': text}]}, config=me)
    display(Markdown(result['messages'][-1].content))   # renders tables and bold text, and never truncates

ask("I have a budget of $5,000 and I prefer low risk. I'm thinking about Apple.")

**Apple (AAPL) – Research Summary**

- **Current price:** $354.07 per share【get_snapshot】  
- **52‑week range:** $243.42 – $360.83【get_snapshot】  
- **Trailing P/E:** 39.43; Forward P/E: 33.49【get_snapshot】  
- **Market cap:** $5.15 trillion (large‑cap, Technology sector)【get_snapshot】  
- **6‑month price performance:** Started at $307.83, ended at $354.07 – **+15.0%**【get_price_trend】  
- **S&P 500 6‑month performance:** Started at 7,519.12, ended at 7,201.09 – **‑4.2%**【get_market_trend】

**Budget Fit**  
With $5,000 you could purchase about **14 shares** ( $5,000 ÷ $354.07 ≈ 14.1 ) of Apple, or the equivalent in fractional shares.  

**Risk Profile**  
Apple is a large‑cap, cash‑rich company with a diversified product line and a strong brand, which generally places it in a lower‑risk tier relative to smaller, more volatile stocks. However, it remains an equity investment, so price can still fluctuate (e.g., a 15% rise over the past six months, while the broader market fell 4.2%).  

**Key Takeaways**  
- Apple’s recent outperformance versus the S&P 500 suggests relative strength.  
- The high market cap and solid earnings multiples (PE ~39) indicate a mature, stable business, aligning with a low‑risk preference.  
- Your $5,000 budget would cover a modest position (≈14 whole shares) or a larger fractional stake.  

*This is research, not financial advice.*

In [10]:
ask('How many shares could I afford, and does that fit my risk level?')

**Affordable shares**

- Current price: $354.07 per share【get_snapshot】  
- $5,000 ÷ $354.07 ≈ **14.12 shares**  

You could purchase **14 whole shares**, costing $4,956.98 and leaving about **$43** cash uninvested. If your broker allows fractional shares, you could invest the full $5,000 for **14.12 shares**.

**Risk fit**

Apple is a large‑cap, cash‑rich technology company with a market cap of $5.15 trillion【get_snapshot】. Its 6‑month price rose **15 %** while the S&P 500 fell **4.2 %**【get_price_trend】【get_market_trend】, indicating relative strength. Large‑cap stocks like Apple are generally considered **lower‑risk** than small‑cap or highly speculative equities, though they still carry normal equity market risk (price can move up or down).

**Conclusion**

- **Affordability:** 14 whole shares (or 14.12 fractional shares) fit comfortably within your $5,000 budget.  
- **Risk alignment:** Apple’s size, strong balance sheet, and recent outperformance suggest it aligns with a **low‑risk** preference, though it is not risk‑free.

*This is research, not financial advice.*

In [11]:
ask('What about Tesla instead? Compare it with what we just looked at.')

**Tesla (TSLA) – Snapshot**

- **Current price:** $384.68 per share【get_snapshot】  
- **52‑week range:** $297.38 – $498.83【get_snapshot】  
- **Trailing P/E:** 422.7; Forward P/E: 177.8【get_snapshot】 (very high, indicating growth‑oriented valuation)  
- **Market cap:** $1.52 trillion (large‑cap, Consumer Cyclical)【get_snapshot】  
- **6‑month price performance:** Started at $433.59, ended at $384.68 – **‑11.3%**【get_price_trend】  

**Affordability with $5,000**

- $5,000 ÷ $384.68 ≈ **13.0 shares**.  
- You could buy **13 whole shares** for $5,000 ≈ $5,000 (13 × $384.68 = $5,000.84, so a fractional share would be needed for exact $5,000).  

**Risk comparison**

| Metric | Apple (AAPL) | Tesla (TSLA) |
|--------|--------------|--------------|
| Sector | Technology (large‑cap) | Consumer Cyclical (large‑cap) |
| 6‑mo price change | **+15.0%** vs S&P 500 ‑4.2%【get_price_trend】【get_market_trend】 | **‑11.3%** (under‑performed market) |
| Trailing P/E | 39.4 | 422.7 |
| Valuation style | Mature, earnings‑driven | High‑growth, earnings‑light |
| Typical risk level | Lower‑risk for a large‑cap equity | Higher‑risk due to growth valuation and cyclical exposure |

**Fit for your low‑risk preference**

- **Apple**: Large‑cap, solid earnings, outperformed the market recently – aligns better with a low‑risk stance.  
- **Tesla**: Very high P/E, recent price decline, and cyclical exposure make it **higher risk** despite its size.

**Bottom line**

- **Shares you can afford:** ~14 Apple shares vs ~13 Tesla shares with $5,000.  
- **Risk match:** Apple fits your low‑risk preference more closely; Tesla is considerably riskier.

*This is research, not financial advice.*

## Important limit
`InMemorySaver` keeps memory **in the notebook's memory only**. If you restart the kernel, everything is gone. A real app uses a saver that writes to a database (LangGraph has SQLite and Postgres versions). The code is the same: only the checkpointer line changes.

**Another limit:** long conversations get expensive and eventually do not fit in the model's context window. We will deal with that in Lesson 6.

## Exercise
1. In Step 5, ask *"What is my budget?"* in a **new** thread id. Does it know?
2. Run `advisor.get_state(me)` and count how many messages the thread holds after three questions. Why so many?
3. Restart the kernel (Restart button at the top), run Step 5 again, and ask the budget question. What happened to the memory?

In [ ]:
# your code here